## Validate final_delivery_enriched_clean.csv by inspecting shape, null values, number of duplicates, and data types

In [1]:
import pandas as pd

df = pd.read_csv("final_delivery_enriched_clean.csv")

print(df.shape)

print(df.isnull().sum())

print(df.duplicated(subset='order_id').sum())

print(df.dtypes)

(95824, 23)
order_id                       0
customer_state                 0
review_score                   0
has_review_flag                0
num_items                      0
num_sellers                    0
num_products                   0
num_product_categories         0
primary_product_category    1343
total_order_value              0
avg_product_weight_g          16
avg_product_volume_cm3        16
primary_payment_type           1
total_installments             1
total_payment_value            1
delivery_days                  0
delay_days                     0
delay_category                 0
delay_bucket                   0
seller_complexity              0
category_complexity            0
severe_late_flag               0
low_review_flag                0
dtype: int64
0
order_id                     object
customer_state               object
review_score                float64
has_review_flag               int64
num_items                     int64
num_sellers                   int6

In [2]:
df['primary_product_category'] = (df['primary_product_category'].fillna('unknown')) # Fill null values in primary_product_category with 'unknown'

df['primary_payment_type'] = (df['primary_payment_type'].fillna('unknown')) # Fill null values in primary_payment_type with 'unknown'

## Check category quality

In [3]:
print(df['primary_product_category'].value_counts().head(25))

print(df['primary_payment_type'].value_counts())

print(df['customer_state'].value_counts().head())

primary_product_category
bed_bath_table                     8997
health_beauty                      8559
sports_leisure                     7475
computers_accessories              6455
furniture_decor                    6168
housewares                         5685
watches_gifts                      5452
telephony                          4066
toys                               3772
auto                               3756
cool_stuff                         3505
garden_tools                       3400
perfumery                          3069
baby                               2697
electronics                        2483
stationery                         2243
fashion_bags_accessories           1804
pet_shop                           1679
unknown                            1343
office_furniture                   1242
luggage_accessories                1008
consoles_games                     1003
home_appliances                     744
construction_tools_construction     719
musical_instrum

## More Quality Assurance checks

In [4]:
print(df.shape)

print(df['order_id'].nunique())

(95824, 23)
95824


In [12]:
print(df[['delivery_days', 'delay_days', 'severe_late_flag']].describe())

       delivery_days    delay_days  severe_late_flag
count   95824.000000  95824.000000      95824.000000
mean       12.052273    -10.993018          0.052440
std         9.466046      9.950629          0.222914
min         0.000000   -146.000000          0.000000
25%         6.000000    -16.000000          0.000000
50%        10.000000    -11.000000          0.000000
75%        15.000000     -6.000000          0.000000
max       208.000000    188.000000          1.000000


In [6]:
print(
    df.groupby('primary_product_category')['delay_days']
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

primary_product_category
arts_and_craftmanship                -5.476190
furniture_mattress_and_upholstery    -6.297297
home_comfort_2                       -7.136364
home_confort                         -8.804124
food                                 -8.907407
audio                                -9.132743
cine_photo                           -9.491803
fashion_underwear_beach              -9.655172
drinks                               -9.978571
electronics                         -10.071285
construction_tools_construction     -10.139082
books_imported                      -10.260000
books_technical                     -10.270916
agro_industry_and_commerce          -10.310734
auto                                -10.313898
Name: delay_days, dtype: float64


## Drill down to specific product categories, exploring what product categories meaningfully drive customer dissatisfaction

In [7]:
print(
    df.groupby('primary_product_category')['severe_late_flag']
    .mean()
    .sort_values(ascending=False)
)

primary_product_category
home_comfort_2                       0.136364
audio                                0.109145
furniture_mattress_and_upholstery    0.108108
fashion_underwear_beach              0.094828
home_confort                         0.082474
                                       ...   
diapers_and_hygiene                  0.000000
party_supplies                       0.000000
fashion_childrens_clothes            0.000000
cds_dvds_musicals                    0.000000
la_cuisine                           0.000000
Name: severe_late_flag, Length: 72, dtype: float64


In [8]:
print(
    df.groupby('primary_product_category')
    .agg({
        'order_id':'count',
        'severe_late_flag':'mean',
        'review_score':'mean'
    })
    .sort_values('severe_late_flag', ascending=False)
)

# The insight obtained: we can't justifiably conclude that the severe lateness of orders from the home_comfort_2 product category is a driver of customer dissatisfaction because there's only 22 orders.

                                   order_id  severe_late_flag  review_score
primary_product_category                                                   
home_comfort_2                           22          0.136364      3.772727
audio                                   339          0.109145      3.851032
furniture_mattress_and_upholstery        37          0.108108      3.891892
fashion_underwear_beach                 116          0.094828      4.008621
home_confort                            388          0.082474      3.902062
...                                     ...               ...           ...
diapers_and_hygiene                      25          0.000000      3.960000
party_supplies                           38          0.000000      4.078947
fashion_childrens_clothes                 7          0.000000      5.000000
cds_dvds_musicals                        12          0.000000      4.666667
la_cuisine                               12          0.000000      4.250000

[72 rows x 

### Continue to analyze product categories that have a meaningful (high severe late %) impact on customer dissatisfaction, filtering for order counts greater than or equal to 300

In [9]:
print(
    df.groupby('primary_product_category')
    .agg({
        'order_id':'count',
        'severe_late_flag':'mean',
        'review_score':'mean'
    })
    .query('order_id >= 300')
    .sort_values('severe_late_flag', ascending=False)
)

                                 order_id  severe_late_flag  review_score
primary_product_category                                                 
audio                                 339          0.109145      3.851032
home_confort                          388          0.082474      3.902062
baby                                 2697          0.065258      4.138116
office_furniture                     1242          0.064412      3.645330
health_beauty                        8559          0.060287      4.235405
construction_tools_construction       719          0.058414      4.132823
bed_bath_table                       8997          0.058242      4.016672
unknown                              1343          0.058079      4.051005
auto                                 3756          0.057774      4.159212
furniture_living_room                 404          0.056931      4.079208
watches_gifts                        5452          0.056860      4.120873
electronics                          2

### Insight from above: 339 orders from the audio category operating with a 10.9% severe lateness rate yields about 37 severely late orders, whereas 2697 orders from the baby category operating with a 6.5% severe lateness rate yields about 175 severely late orders. This indicates that certain product categories may operate with higher risk but yield low impact to customer satisfaction, while other product categories operate at moderate risk levels and yield a high impact to customer satisfaction. We continue this analysis of high-volume product cateogies with above_average severe-late rates and their contribution to customer dissatisfaction in Power BI.

In [10]:
df.to_csv("final_delivery_enriched_clean_final.csv", index=False)